# 01 May 05 Default Best Fixed Baseline

Re-runs the best May 04 default DRLB params as the May 05 control point.

In [5]:
import sys
import json
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess


In [6]:
RUN_NAME = 'may05_default_best_fixed_baseline'
DRLB_PROFILE = 'may05_default_best_fixed'
VERBOSE = False


In [ ]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
config = replace(config, n_trials=1, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'base_drlb_params': base_drlb_params,
    'reference_model_params': reference_model_params,
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}


In [ ]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])


,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
